# MCP Architecture: Who Talks, and What They Can Offer

Every MCP conversation involves exactly three participants — a host, a client, and a
server — and a server relationship can offer three specific things once a connection
exists. This notebook walks through both, using one small real server as the anchor for
everything, and a diagram for every idea so the shape of each concept is visible, not
just described.

**Built with real code in this notebook:** the server itself, and its tools.
**Explained in depth, with diagrams:** the host, the client, the server relationship,
resources, and prompts. Full working code for resources and prompts arrives in a later,
dedicated notebook — this one is about understanding the shape of each idea first.


## Setup

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh        # macOS/Linux
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"   # Windows

uv init mcp-warmup && cd mcp-warmup
uv add fastmcp
```


In [ ]:
server_code = '''
from fastmcp import FastMCP

mcp = FastMCP("Warm-Up Server")

@mcp.tool
def greet(name: str) -> str:
    """Greet someone by name."""
    return f"Hello, {name}! Welcome to MCP."

@mcp.tool
def add(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b

if __name__ == "__main__":
    mcp.run()
'''

with open('server.py', 'w') as f:
    f.write(server_code)

print(server_code)


```bash
uv run fastmcp dev server.py
```

This opens **MCP Inspector** at `http://127.0.0.1:6274` — a browser tool that lets you
connect to any MCP server and try its capabilities directly, without needing a full AI
application involved at all. Click the **Tools** tab, call `greet` with your own name,
then call `add` with two numbers. Notice that the description shown for each tool is
exactly the docstring written in the Python function above — that listing was generated
automatically, not written by hand.

You've now watched two separate programs — Inspector, and `server.py` — communicate and
complete a task together. Everything below names, precisely, what just happened.


## The Three Participants

One picture is worth carrying through this entire course: a universal remote sitting in
a house full of smart gadgets, with a dedicated adapter behind each one, translating the
remote's button presses into whatever signal that specific gadget expects. That picture
maps directly onto MCP's three participants.


```mermaid
flowchart LR
 P["Person"] --> H["Host\n"]
 H --> C["Client\n"]
 C --> S["Server \n"]
```


### The Host

The **host** is the AI application a person actually interacts with — the thing that
receives a question, decides (using the language model underneath it) whether it needs
outside help, and coordinates whatever happens next. In the setup above, MCP Inspector's
own interface was playing this role. In everyday use, this is Claude Desktop, Cursor, or
a custom chatbot someone builds.

The detail worth sitting with: **the host never talks to a server directly.** It
delegates that entirely. If every host had to understand the internal wiring of every
tool it might ever use, adding one new capability would be a real engineering project
every time. Instead, a host only ever needs to know how to *ask* — never how any
individual server does its job internally. That's what lets one AI application connect
to a tool it has never seen before, built by a team it has never spoken to.

In the smart-home picture, the host is the person holding the remote. They never
personally rewire a gadget — they press a button, and something else handles the rest.


### The Server

A **server** is a program that does one specific job — or a closely related family of
jobs — and exposes that capability clearly enough for anything speaking the right
language to find and use it. `server.py`, running in your terminal right now, is a
server. It knows how to greet someone and add two numbers, and nothing else, and that's
completely fine — a good server doesn't need to be a general-purpose assistant, only
reliably good at its one job.

This matters more than it looks. Because a server only has to do one thing well, it can
be built by whoever actually understands that thing best — often the company that owns
the underlying service. A team running a version-control platform is far better placed to
build a correct, secure connector to their own system than every individual AI company
trying to reverse-engineer it separately. Once a shared protocol exists, a service only
has to build its own official server *once*, and every AI application that speaks the
protocol can use it.

In the smart-home picture, the server is the actual gadget. It doesn't know or care which
brand of remote is pointing at it — it only needs to expose its buttons in a shared
format.


### The Client

This is the piece most explanations rush past, and it's worth slowing down for, because
it's the actual reason MCP's architecture looks the way it does. The host doesn't talk
to a server directly — it creates a dedicated **client** for each server it wants to
use, and that client maintains a private, one-to-one connection with exactly one server.
A host connected to five servers is running five separate clients, each with its own
dedicated line — never one shared connection to all five.

In the setup above, the connection Inspector opened the instant it launched — that
handshake, that open channel to `server.py` — was a client doing its job.


```mermaid
flowchart TB
 Host["Host (AI Application)"] --> C1[Client 1]
 Host --> C2[Client 2]
 Host --> C3[Client 3]
 C1 -->|dedicated connection| S1[(Server A)]
 C2 -->|dedicated connection| S2[(Server B)]
 C3 -->|dedicated connection| S3[(Server C)]
```

> Each MCP client maintains a dedicated, 1:1 connection with its corresponding MCP
> server. Local servers typically serve a single client, whereas remote servers
> typically serve many clients at once.
> *(Model Context Protocol official documentation)*


**Why this design, instead of one shared connection to every server?** Four concrete
reasons, each worth understanding on its own terms rather than as a list to memorize.


```mermaid
flowchart LR
 D[Decoupling] --- S[Safety] --- SC[Scalability] --- P[Parallelism]
```

**Decoupling.** Because each client only ever talks to one server, changing or updating
how one server works never requires touching how the host talks to any other server.
The connections are entirely independent.

**Safety.** A crash or serious bug in one server's connection stays contained to that
one connection. You can prove this yourself: with `server.py` running, switch to its
terminal and press Ctrl+C. Look at what happens in Inspector — it doesn't crash or
freeze, it simply shows that one connection as closed. If Inspector had five servers
connected and you killed one, the other four would be completely unaffected. That calm,
contained failure is a direct consequence of dedicated connections rather than a shared
one.

**Scalability.** If one particular server suddenly needs to handle much heavier use,
that growth is entirely local to that one client-server pair — none of the host's other
connections need to change to accommodate it.

**Parallelism.** A host with three live connections can talk to all three servers at the
same time, rather than being forced to wait for one slow server before even starting the
next. A shared connection would force them to effectively take turns.


## What a Server Can Offer: Three Primitives

Once a client and a server are connected, the protocol defines exactly three things a
server is allowed to give that client. Only the first has real running code in this
notebook — the other two are explained fully, with diagrams, and get their own
dedicated build in a later notebook.


```mermaid
flowchart TB
 subgraph Server["A Server Can Offer..."]
 T["Tools\nsomething that DOES"]
 R["Resources\nsomething to READ"]
 P["Prompts\na FORM to fill in"]
 end
```


### Tools — proven above

A **tool** is an executable action — something with a real effect when called, not just
information being returned. `greet` and `add`, which you already called, are both
tools. Calling `add` didn't retrieve a fact that already existed somewhere; it ran a
small piece of logic and produced a new result.

Tools matter because they're the one primitive that lets an AI application actually *do*
something, rather than only ever describing what doing something would involve. Without
tools, an AI can tell you in perfect detail what sending an email would look like. With a
tool, it can actually send one.


### Resources — the read-only layer

A **resource** is read-only reference data a server makes available — something to look
at, with no side effects and no action performed. Where a tool *does* something, a
resource simply *is* something, sitting there for anyone connected to read.


```mermaid
flowchart LR
 C1[Client 1] -->|reads| Res[(Shared Resource)]
 C2[Client 2] -->|reads| Res
 C3[Client 3] -->|reads| Res
```

In the smart-home picture, a resource is the small digital display on a smart
thermostat — you don't ask it the current temperature in some active way, you just
glance, because it's always sitting there for anyone who looks.

Resources solve a specific, easy-to-miss problem: without one shared resource, every
client that needs the same reference data would have to hard-code its own copy — and
over time, as one copy gets updated and another doesn't, the two quietly drift apart
until something breaks. A single resource, read fresh by everyone who needs it, never has
that problem, because there's only ever one copy of the truth. *(The full working code
for adding a resource to a server is covered in a later, dedicated notebook.)*


### Prompts — a form, not data

A **prompt** is a reusable template that shapes *how* a request gets phrased — not data,
and not an action. It's closer to a form with blank fields, provided by the server, that
guides a request toward being consistent and complete instead of improvised from scratch
every time.


```mermaid
flowchart LR
 A["No template\n'customer unhappy, refund maybe'"] -->|server provides a template| B["With a template\nIssue / Tried / Sentiment / Next step"]
```

Picture an AI logging a customer support escalation with no guidance at all. Left to its
own judgment, it might write something as thin as *"customer unhappy, refund maybe"* —
technically a record, but not useful to a support agent who has to act on it. Now picture
a server that hands it a template requiring four specific fields every time: what the
issue was, what had already been tried, how the customer seemed to feel, and a
recommended next step. Nothing about the underlying model changed, and the question being
asked didn't change either — only the presence of a form did.

This is the real value of a prompt: consistency that survives *who's* asking. Every AI
application that connects to this server and uses this prompt produces escalations with
the same structure, because that structure lives once, on the server, instead of being
separately reinvented — possibly differently — inside every application that happens to
use it. *(The full working code for adding a prompt to a server is covered in a later,
dedicated notebook.)*


## Summary

```mermaid
mindmap
 root((MCP<br/>Architecture))
 Host
 the AI app,<br/>never talks to a<br/>server directly
 Client
 one dedicated<br/>connection per server
 Server
 one focused<br/>capability
 Offers
 Tools -- built today
 Resources -- next notebook
 Prompts -- next notebook
```

Three participants, always in the same shape: a host that coordinates, a dedicated client
per server, and a server that does one job well. And three things a server relationship
can offer: tools to act, resources to read, and prompts to structure a request
consistently. Tools are already real, in the server you built at the top of this
notebook. Resources and prompts are understood in full — what they are, why they matter,
and what advantage each one buys — and get their real code in the notebook that follows.

**Next:** the actual language these three participants speak to each other — what a
message looks like on the wire, and how a connection knows when it has started and when
it has ended.
